In [1]:
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sklearn.metrics.pairwise import cosine_similarity, cosine_distances
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from urllib.parse import urlparse
from tempfile import mkdtemp
from typing import Union
from enum import Enum
import numpy as np
import logging
import pickle
import shutil
import json
import csv
import os
import re


In [2]:
DATA_FILE = '/archive/Downloads/data.json'
METADATA_FILE = '/archive/Downloads/metadata_full.tsv'
CSVS_DIR = '/archive/Downloads/phishtank'

In [3]:
with open(DATA_FILE, 'r') as f:
    data = json.load(f)

In [4]:
clusters = {}
with open(METADATA_FILE, 'r') as f:
    next(f)
    for line in f:
        line = line.rstrip().split('\t')
        hash = line[0]
        cluster = line[1]
        
        if cluster not in clusters:
            clusters[cluster] = []

        clusters[cluster].append(hash)

In [5]:
phishing_lures = {}

from dateutil.parser import parse

for file in os.listdir(CSVS_DIR):
    if not file.endswith('.csv'):
        continue

    with open(os.path.join(CSVS_DIR, file), 'r') as f:
        reader = csv.reader(f)
        next(reader)
        for row in reader:
            url = row[1]
            submission_date = row[3]

            submission_date = parse(submission_date)
            # submission_date = f"{submission_date.year}-{submission_date.month:02d}-{submission_date.day:02d}"

            if url in phishing_lures:
                break

            phishing_lures[url] = submission_date

In [6]:
cluster_dates = {}

for cluster_idx in clusters:
    sample_dates = {}
    min_date = None
    max_date = None

    for sample in clusters[cluster_idx]:
        domains = []
        for segment in data[sample].values():
            domains.append(segment['domain'])
        for domain in domains:
            if domain in phishing_lures:
                sample_dates[sample] = phishing_lures[domain]

                if min_date is None or phishing_lures[domain] < min_date:
                    min_date = phishing_lures[domain]

                if max_date is None or phishing_lures[domain] > max_date:
                    max_date = phishing_lures[domain]

                break
    
    if len(sample_dates) == 0:
        continue

    cluster_dates[cluster_idx] = (max_date - min_date).days

In [7]:
cluster_dates

{'0': 0,
 '1': 260,
 '2': 363,
 '4': 0,
 '5': 185,
 '6': 0,
 '8': 41,
 '10': 0,
 '14': 754,
 '18': 0,
 '23': 0,
 '25': 485,
 '26': 0,
 '27': 0,
 '29': 903,
 '30': 534,
 '32': 0,
 '36': 0,
 '37': 481,
 '38': 283,
 '39': 0,
 '42': 0,
 '45': 0,
 '48': 0,
 '52': 413,
 '56': 620,
 '58': 0,
 '60': 0,
 '61': 0,
 '65': 68,
 '66': 0,
 '67': 243,
 '69': 0,
 '70': 0,
 '72': 0,
 '73': 0,
 '76': 682,
 '79': 0,
 '80': 0,
 '81': 0,
 '82': 0,
 '86': 6,
 '88': 0,
 '89': 670,
 '92': 0,
 '96': 0,
 '99': 0,
 '103': 0,
 '107': 767,
 '108': 0,
 '111': 0,
 '112': 0,
 '113': 0,
 '117': 0,
 '119': 0,
 '120': 0,
 '126': 0,
 '127': 0,
 '131': 0,
 '133': 0,
 '134': 41,
 '135': 0,
 '140': 0,
 '141': 489,
 '143': 251,
 '144': 0,
 '145': 0,
 '147': 0,
 '148': 0,
 '149': 0,
 '151': 0,
 '154': 26,
 '159': 0,
 '162': 131,
 '163': 676,
 '165': 0,
 '167': 0,
 '170': 7,
 '172': 721,
 '173': 0,
 '174': 0,
 '175': 0,
 '176': 757,
 '177': 0,
 '178': 375,
 '179': 688,
 '180': 0,
 '181': 23,
 '182': 229,
 '183': 721,
 '184': 0

In [8]:
with open('cluster_date_intervals.json', 'w') as f:
    json.dump(cluster_dates, f, indent=4)

In [47]:
sample_dates

{' e7a1804041ebe3c4': '2024-04-16T21:30:07+00:00',
 ' abf5c3057c7aba8b': '2024-01-04T14:23:07+00:00',
 ' 7bcf8e3db44f2553': '2023-08-24T03:01:45+00:00',
 ' 69862ceaa6b5a5d0': '2024-03-28T16:11:03+00:00',
 ' 684006d871586bff': '2023-07-31T12:01:18+00:00',
 ' c5bfe2e52a5579d6': '2024-02-22T13:33:14+00:00',
 ' 6a7cae04ca8d3c1b': '2024-02-15T19:30:03+00:00',
 ' 498a00a724640d37': '2023-08-24T12:10:37+00:00',
 ' e2b8824b9b638cbf': '2024-02-29T00:11:07+00:00',
 ' b005b91d504b39c4': '2024-03-15T00:31:38+00:00'}